In [2]:
import re
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from scipy.sparse import hstack, csr_matrix
from xgboost import XGBClassifier

In [3]:
# Load  and augement data

PROC = "../Data/processed" 
SEED = 42 

# real notes (from test split)
with open(f"{PROC}/test.txt", encoding="utf-8") as fh:
    real_texts = [t.strip() for t in fh.read().split("\n<|endoftext|>\n") if t.strip()]

# synthetic sources
sources = {
    "qwen_freegen":  pd.read_csv(f"{PROC}/synthetic_notes_qwen.csv")["generated_text"],
    "llama_freegen": pd.read_csv(f"{PROC}/synthetic_notes_llama.csv")["generated_text"],
    "qwen_rewrite":  pd.read_csv(f"{PROC}/synthetic_notes_qwen_rewrite.csv")["generated_text"],
    "llama_rewrite": pd.read_csv(f"{PROC}/synthetic_notes_llama_rewrite.csv")["generated_text"],
}

# collect synthetic rows
syn_rows = []
for src, series in sources.items():
    for t in series.astype(str):
        if t.strip():
            syn_rows.append({"text": t, "source": src, "label": 1})

n_syn = len(syn_rows)

# sample an equal number of real notes for a balanced binary dataset
real_sample = pd.Series(real_texts).sample(n=n_syn, random_state=SEED).tolist()
real_rows = [{"text": t, "source": "real", "label": 0} for t in real_sample]

combined = pd.DataFrame(real_rows + syn_rows).sample(frac=1, random_state=SEED).reset_index(drop=True)
combined.to_csv(f"{PROC}/augmented_dataset.csv", index=False)

print(combined["source"].value_counts())
print(f"\nBinary balance: {combined['label'].value_counts().to_dict()}")
print(f"Total: {len(combined)} notes")

source
real             4000
qwen_freegen     1000
llama_rewrite    1000
llama_freegen    1000
qwen_rewrite     1000
Name: count, dtype: int64

Binary balance: {0: 4000, 1: 4000}
Total: 8000 notes


In [4]:
# Run xgboost on this 
data = combined.dropna(subset=["text"])
print(f"Binary balance: {data['label'].value_counts().to_dict()}\n")

Binary balance: {0: 4000, 1: 4000}



In [ ]:
# Feature engineering and cleaning
def strip_structure(text):
    """Remove de-identification and MIMIC header scaffolding so the classifier
    is forced onto clinical prose rather than formatting artifacts."""
    text = str(text)
    text = re.sub(r'<?PHI\w*', ' ', text)
    text = re.sub(r'\*\*', '', text)          # remove markdown bold
    text = re.sub(r'#{1,6}\s*', '', text)

    header_pattern = (
        r'(?im)^\s*(Name|Unit No|Unit Number|Admission Date|Discharge Date|Date of Birth|Sex|'
        r'Gender|Service|Attending|Attending Physician|Allergies|Allergy|Followup Instructions|'
        r'Discharge Disposition):.*$'
    )
    text = re.sub(header_pattern, ' ', text)

    clinical = (r'(?i)(Chief Complaint|History of Present Illness|Past Medical History|'
                r'Physical Exam|Social History|Family History|Pertinent Results|'
                r'Brief Hospital Course|Discharge Medications|Discharge Instructions|'
                r'Discharge Diagnosis|Discharge Condition|Medications on Admission|'
                r'Major Surgical or Invasive Procedure)\s*:?')
    
    text = re.sub(clinical, ' ', text)
    # Bare header-label words that survive as fragments (sex service, unit no, birth sex...)

    labels = (r'(?i)\b(Sex|Service|Unit No|Unit Number|Admission Date|Discharge Date|'
              r'Date of Birth|Birth|Attending|Disposition|Followup|Admission)\b')
    text = re.sub(labels, ' ', text)
    text = re.sub(r'\d{1,2}/\d{1,2}/\d{2,4}', ' ', text)
    text = re.sub(r'\s+', ' ', text)   # tidy the whitespace the removals leave behind
    return text

CONNECTORS = ["however","furthermore","additionally","moreover","therefore",
              "consequently","subsequently","notably","overall","importantly","specifically"]

def extract_features(text):
    text=str(text); sents=[s for s in re.split(r'[.!?]',text) if s.strip()]
    sl=[len(s.split()) for s in sents]; words=text.split(); nw=len(words) or 1; nc=len(text) or 1; tl=text.lower()
    f={}
    f["word_count"]=len(words); f["sentence_count"]=len(sents)
    f["avg_sentence_length"]=np.mean(sl) if sl else 0
    f["sentence_length_std"]=np.std(sl) if len(sl)>1 else 0
    f["sentence_length_cv"]=f["sentence_length_std"]/f["avg_sentence_length"] if f["avg_sentence_length"]>0 else 0
    f["vocab_richness"]=len(set(w.lower() for w in words))/nw
    f["uppercase_word_ratio"]=sum(1 for w in words if w.isupper())/nw
    f["comma_density"]=text.count(",")/nc*1000; f["colon_density"]=text.count(":")/nc*1000
    f["semicolon_density"]=text.count(";")/nc*1000; f["paren_density"]=text.count("(")/nc*1000
    syll=sum(max(1,len(re.findall(r'[aeiouyAEIOUY]+',w))) for w in words)
    f["flesch"]=206.835-1.015*(nw/max(1,len(sents)))-84.6*(syll/nw)
    f["connector_density"]=sum(tl.count(c) for c in CONNECTORS)/nw*100
    return f

In [6]:
texts = np.array(data["text"].tolist(), dtype=object)
labels = data["label"].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
aucs, accs, f1s = [], [], []
for tr, te in skf.split(texts, labels):
    trx, tex = texts[tr], texts[te]; trY, teY = labels[tr], labels[te]
    trc = [strip_structure(t) for t in trx]; tec = [strip_structure(t) for t in tex]
    tfidf = TfidfVectorizer(max_features=500, min_df=5, ngram_range=(1,2), sublinear_tf=True)
    Xtr_t = tfidf.fit_transform(trc); Xte_t = tfidf.transform(tec)
    htr = pd.DataFrame([extract_features(t) for t in trx]); hte = pd.DataFrame([extract_features(t) for t in tex])
    Xtr = hstack([Xtr_t, csr_matrix(htr.values)]); Xte = hstack([Xte_t, csr_matrix(hte.values)])
    clf = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8,
                        colsample_bytree=0.3, reg_alpha=1.0, reg_lambda=2.0,
                        eval_metric="logloss", random_state=42)
    clf.fit(Xtr, trY)
    p = clf.predict_proba(Xte)[:,1]; pred = (p>=0.5).astype(int)
    aucs.append(roc_auc_score(teY,p)); accs.append(accuracy_score(teY,pred)); f1s.append(f1_score(teY,pred))

print("Pooled real-vs-synthetic (5-fold CV)")
print(f"  AUC      {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
print(f"  Accuracy {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"  F1       {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")

Pooled real-vs-synthetic (5-fold CV)
  AUC      1.0000 ± 0.0000
  Accuracy 0.9999 ± 0.0002
  F1       0.9999 ± 0.0002


In [7]:
# ---- single clean split for the XAI bundle (CV above stays your reported metric) ----
tr_x, te_x, tr_y, te_y = train_test_split(
    texts, labels, test_size=0.2, random_state=SEED, stratify=labels)

tr_clean = [strip_structure(t) for t in tr_x]
te_clean = [strip_structure(t) for t in te_x]

tfidf = TfidfVectorizer(max_features=500, min_df=5, ngram_range=(1,2), sublinear_tf=True)
Xtr_t = tfidf.fit_transform(tr_clean)
Xte_t = tfidf.transform(te_clean)

htr = pd.DataFrame([extract_features(t) for t in tr_x])   # hand features on RAW text
hte = pd.DataFrame([extract_features(t) for t in te_x])
hand_names = list(htr.columns)
feat_names = tfidf.get_feature_names_out().tolist() + hand_names

Xtr = hstack([Xtr_t, csr_matrix(htr.values)]).tocsr()
Xte = hstack([Xte_t, csr_matrix(hte.values)]).tocsr()

clf = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8,
                    colsample_bytree=0.3, reg_alpha=1.0, reg_lambda=2.0,
                    eval_metric="logloss", random_state=SEED)
clf.fit(Xtr, tr_y)
prob = clf.predict_proba(Xte)[:,1]
pred = (prob >= 0.5).astype(int)
print(f"[augmented] AUC {roc_auc_score(te_y, prob):.4f}  Acc {accuracy_score(te_y, pred):.4f}")

joblib.dump({
    "clf": clf, "tfidf": tfidf,
    "feat_names": feat_names, "hand_names": hand_names,
    "Xte": Xte, "te_x": te_x, "te_y": list(te_y),
    "hte": hte, "htr": htr, "tr_y": list(tr_y),
    "prob": prob, "pred": pred,
}, "xgb_bundle_augmented.joblib")
print("Saved xgb_bundle_augmented.joblib")

[augmented] AUC 1.0000  Acc 1.0000
Saved xgb_bundle_augmented.joblib
